# Cálculo de Caída de Tensión (12V/24V)
Dimensionamiento de la sección de cables marinos para evitar caídas de tensión.

Este simulador está diseñado para fines educativos. **No lo utilices para la navegación real.**

<a href="https://colab.research.google.com/github/jorgejuan007/Nautica/blob/main/simulaciones/62_simulador_panel_electrico_12v.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt

def calcular_seccion(voltaje, amperios, metros_ida_vuelta, caida_permitida):
    # Resistividad del cobre a 20ºC (Ohm * mm2 / m)
    rho_cobre = 0.0171
    
    # Voltaje de caída máxima permitida
    caida_voltios = voltaje * (caida_permitida / 100)
    
    # Fórmula: Seccion (mm2) = (rho * L * I) / V_caida
    seccion_minima = (rho_cobre * metros_ida_vuelta * amperios) / caida_voltios
    
    secciones_comerciales = [1.5, 2.5, 4, 6, 10, 16, 25, 35, 50, 70, 95]
    seccion_recomendada = min([s for s in secciones_comerciales if s >= seccion_minima], default=95)
    
    if seccion_minima > 95:
        print("⚠️ PRECAUCIÓN: Sección necesaria extremadamente gruesa. Considere aumentar el voltaje del sistema (ej. 24V o 48V) o reducir la longitud del cable.")
        seccion_recomendada = "Mayor a 95"
        
    print(f"Sistema: {voltaje}V")
    print(f"Consumo: {amperios} Amperios ({voltaje * amperios} Vatios)")
    print(f"Longitud total del cableado (ida + vuelta): {metros_ida_vuelta} metros")
    print(f"Caída de tensión permitida: {caida_permitida}% ({caida_voltios:.2f}V)")
    print(f"\nSección teórica mínima calculada: {seccion_minima:.2f} mm²")
    print(f"Sección comercial recomendada: {seccion_recomendada} mm²")
    
    # Gráfico de caída de tensión vs sección
    secciones = np.array([1.5, 2.5, 4, 6, 10, 16, 25, 35, 50, 70, 95])
    caidas = (rho_cobre * metros_ida_vuelta * amperios) / secciones
    porcentajes = (caidas / voltaje) * 100
    
    plt.figure(figsize=(8,4))
    plt.plot(secciones, porcentajes, 'b-o')
    plt.axhline(caida_permitida, color='r', linestyle='--', label=f'Límite {caida_permitida}%')
    plt.axvline(seccion_minima, color='g', linestyle=':', label=f'Mínimo {seccion_minima:.1f}mm²')
    plt.title('Caída de tensión vs. Sección del Cable (12V/24V)')
    plt.xlabel('Sección del Cable (mm²)')
    plt.ylabel('Caída de Tensión (%)')
    plt.legend()
    plt.grid(True)
    plt.show()

v = widgets.Dropdown(options=[12, 24, 48], value=12, description='Voltaje:')
a = widgets.FloatSlider(value=10, min=1, max=200, step=1, description='Amperios (A):')
m = widgets.FloatSlider(value=10, min=1, max=50, step=1, description='Metros (i+v):')
c = widgets.Dropdown(options=[('3% (Equipos Críticos)', 3), ('10% (Iluminación general)', 10)], value=3, description='Caída Max:')

out = widgets.interactive_output(calcular_seccion, {'voltaje': v, 'amperios': a, 'metros_ida_vuelta': m, 'caida_permitida': c})
display(widgets.VBox([v, a, m, c, out]))
